<a href="https://colab.research.google.com/github/iDurugkar/practice-2026/blob/main/TorchCode/38_grpo_loss.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/38_grpo_loss.ipynb)

# 🔴 Hard: GRPO Loss

Implement the **Group Relative Policy Optimization (GRPO)** loss — a group-wise, baseline-subtracted REINFORCE objective commonly used in RLAIF (reinforcement learning from AI feedback).

Given a batch of log-probabilities, scalar rewards, and group ids (one group per prompt), define the within-group normalized advantages:

$$A_i = \frac{r_i - \bar r_{g(i)}}{\text{std}_{g(i)} + \epsilon}$$

where \(\bar r_{g(i)}\) and \(\text{std}_{g(i)}\) are the mean and standard deviation of rewards in the group of example \(i\).

The GRPO loss is then the negative advantage-weighted log-probability:

$$\mathcal{L}_{\text{GRPO}} = -\mathbb{E}_i \big[\,\text{stop\_grad}(A_i)\, \log \pi_\theta(y_i)\big].$$

### Signature
```python
from torch import Tensor

def grpo_loss(logps: Tensor, rewards: Tensor, group_ids: Tensor,
              eps: float = 1e-5) -> Tensor:
    """GRPO loss over a batch.

    logps: (B,) policy log-probs for each sampled response
    rewards: (B,) scalar rewards for each response
    group_ids: (B,) integers, same id = same prompt/group
    returns: scalar loss (Tensor)
    """
```

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.7 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn.functional as F

In [12]:
# ✏️ YOUR IMPLEMENTATION HERE

from torch import Tensor

def grpo_loss(logps: Tensor, rewards: Tensor, group_ids: Tensor,
              eps: float = 1e-5) -> Tensor:
  # compute normalized advantages per group and return -mean(adv.detach() * logps)
  gids = group_ids.unique()
  advantages = torch.empty_like(rewards)

  for id in gids:
    mask = group_ids == id
    r_g = rewards[mask]
    mean_r = r_g.mean()
    std_r = r_g.std(unbiased=False)

    advantages[mask] = ((r_g - mean_r) / (std_r + eps))
  adv = advantages.detach()
  return -(adv * logps).mean()


In [13]:
# 🧪 Debug
logps = torch.tensor([0.0, -0.5, -1.0, -1.5])
rewards = torch.tensor([1.0, 0.8, 0.2, 0.0])
group_ids = torch.tensor([0, 0, 1, 1])
print('Loss:', grpo_loss(logps, rewards, group_ids).item())

Loss: -0.24997496604919434


In [14]:
# ✅ SUBMIT
from torch_judge import check
check('grpo_loss')


🧪 Testing: GRPO (Group Relative Policy Optimization) Loss (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Basic shape & type (1.6ms)
  ✅ [2/4] Numeric check vs reference (2.1ms)
  ✅ [3/4] Gradient flows to logps only (1.0ms)
  ✅ [4/4] Group-wise normalization (0.7ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (5.5ms total)
  Progress saved. Run status() to see your dashboard.



In [3]:
from torch_judge import hint
hint('grpo_loss')


💡 Hint for GRPO (Group Relative Policy Optimization) Loss:
   Per group, normalize rewards: A_i = (r_i - mean_g) / (std_g + eps). Detach A_i from graph, then return -mean(A_i * logps).

